# v17 · Address size across layers: shipped charset vs. per-layer restricted charset vs. compact format
The calendrical index as described for this paper: chunk entries `{a, n, t}` → JSON day → JSON month, each layer addressed with the same class. Code below is extracted verbatim from the audit script `bench_layers2.py` (only the corpus source is replaced by the fixed 4000-character sample).

In [1]:
import importlib.util, os, sys, time, math, hashlib, json
from collections import Counter
PROJ = os.environ.get("PROJ", "/home/rendier/Projects/ThePlace")
NB = os.path.join(PROJ, "FourthAgePapers/HyperindexingSystem/notebooks")
CORPUS = open(os.path.join(NB, "corpus_sample.txt"), encoding="utf-8").read()   # fixed 4000-char sample, 97-symbol keyboard set
def load(rel, name="m"):
    """Load a source file from the repo by path, unmodified."""
    spec = importlib.util.spec_from_file_location(name, os.path.join(PROJ, rel))
    m = importlib.util.module_from_spec(spec); spec.loader.exec_module(m); return m
print("corpus:", len(CORPUS), "chars, sha256", hashlib.sha256(CORPUS.encode()).hexdigest()[:16])

corpus: 4000 chars, sha256 3105964b9180566a


In [2]:
hw_mod = load("PtolemyDesktop/Callimachus/HyperWebster-Data-Storage/hyperwebster.py", "hw_mod")
HyperWebster = hw_mod.HyperWebster
DEFAULT = HyperWebster()
import collections, json
_src = open(os.path.join(PROJ, "ContextPlease/claude/scratchpad/2026-09-29_hyperwebster_overhead/bench_layers2.py")).read()
exec(_src[_src.index("def charset_for"):_src.index("def main")])       # build(), restore(), charset_for(), chunk_texts()
print("extracted:", [n for n in ("charset_for", "chunk_texts", "build", "restore") if n in globals()])

extracted: ['charset_for', 'chunk_texts', 'build', 'restore']


In [3]:
text = CORPUS; raw = 8 * len(text); chunk = 200
print("corpus %d chars = %d raw bits | chunk %d | 4 chunks/day | 5 days/month" % (len(text), raw, chunk))
print("%-22s %-10s %-26s %-9s %-7s %s" % ("mode", "|charset|", "chunk/day/month (x raw)", "top ptr", "exact", "secs"))
for mode, compact in (("M0 shipped 97", False), ("B restricted", False), ("A+B compact (hex,;)", True)):
    t0 = time.perf_counter()
    L = build(text, chunk, mode if not compact else "A+B", compact)
    ok = restore(L, compact) == text
    bits = [sum(a.bit_length() for a, _ in l["ents"]) for l in L]
    print("%-22s %-10s %.3f / %.3f / %.3f      %-9.3f %-7s %.1f" % (mode, "/".join(str(len(l["cs"])) for l in L), bits[0]/raw, bits[1]/raw, bits[2]/raw, bits[2]/raw, ok, time.perf_counter()-t0))
    assert ok

corpus 4000 chars = 32000 raw bits | chunk 200 | 4 chunks/day | 5 days/month
mode                   |charset|  chunk/day/month (x raw)    top ptr   exact   secs
M0 shipped 97          97/97/97   0.824 / 1.480 / 2.472      2.472     True    4.8
B restricted           65/27/27   0.752 / 0.981 / 1.188      1.188     True    1.1
A+B compact (hex,;)    65/18/18   0.751 / 0.803 / 0.842      0.842     True    0.5


Every row restores the original text exactly. With the shipped 97-symbol charset the address of each layer is written in the same charset and grows by the JSON overhead each time (≈1.7× per layer here): the top pointer exceeds the size of the corpus. Restricting each layer to the symbols it contains, and using a compact non-JSON layer format, brings it below 1. This is the measured content of the 'layered index' claim: **time per layer is flat; size per layer is not, unless the layer format is chosen for it.**